# TFM: Análisis de Políticas de Sostenibilidad mediante técnicas de Argumentacion Computacional

## Clasificación de relaciones con ollama_chat/llama3.3:70b en poliGPT API

In [1]:
%pip install langchain pymupdf openai openpyxl pandas numpy openpyxl --quiet

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
from typing import List
from pydantic import BaseModel, Field, ValidationError
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import PromptTemplate
from langchain_core.exceptions import OutputParserException
import requests
import json
import re
from openai import OpenAI
import openai
import httpx
import pandas as pd
import numpy as np
import os
import openpyxl

## Input text processing

In [3]:
def classify_relationships(input_dir, prefix, model_name):
    client = OpenAI(
        base_url='https://api.poligpt.upv.es',
        api_key='sk-Icbf-5FyeV0QcLWBC9SNEA'
    )

    for filename in os.listdir(input_dir):
        if filename.endswith(".csv") and prefix in filename:
            filepath = os.path.join(input_dir, filename)
            print(f"Processing {filepath} ...")

            df = pd.read_csv(filepath)

            rel_col = f"rel_{model_name}"
            df[rel_col] = ""

            for idx, row in df.iterrows():
                arg1, arg2 = row["SDGarg1"], row["SDGarg2"]

                prompt = (
                    "Classify the relationship between the following two arguments. "
                    "Return only one label from: Support, Attack, Rephrase, No Relationship.\n\n"
                    f"Argument 1: {arg1}\n"
                    f"Argument 2: {arg2}\n\n"
                    "Label:"
                )

                try:
                    chat_completion = client.chat.completions.create(
                        messages=[
                            {"role": "system", "content": "You are an expert in argument analysis."},
                            {"role": "user", "content": prompt}
                        ],
                        model=model_name,
                        temperature=0,
                    )
                    label = chat_completion.choices[0].message.content.strip()
                    # Check for label 
                    if label not in ["Support", "Attack", "Rephrase", "No Relationship"]:
                        if "support" in label.lower():
                            label = "Support"
                        elif "attack" in label.lower():
                            label = "Attack"
                        elif "rephrase" in label.lower():
                            label = "Rephrase"
                        elif "no relationship" in label.lower():
                            label = "No Relationship"
                        else:
                            print(f"Unexpected output at row {idx}: {label}")
                            label = "No Relationship"
                except Exception as e:
                    print(f"Error processing row {idx}: {e}")
                    label = "No Relationship"

                df.at[idx, rel_col] = label

                if idx % 50 == 0:
                    print(f"  Processed {idx}/{len(df)} rows...")

            
            df.to_csv(filepath, index=False, encoding="utf-8")
            print(f"Saved classified file to {filepath}")



In [ ]:
process_rel_path = "..\\Data\\Relationships No Keywords\\"
model_name="llama"

prefix = 'GLOBAL_SGD2023_qwen2.5-3b'

classify_relationships(process_rel_path, prefix, model_name)

Processing ..\Data\Relationships No Keywords\cross_goalGLOBAL_SGD2023_qwen2.5-3b.csv ...
  Processed 0/11822 rows...


In [ ]:
process_rel_path = "..\\Data\\Relationships No Keywords\\"
model_name="llama"

prefix = 'intra_goalGLOBAL_SGD2023_gemma3-4b'

classify_relationships(process_rel_path, prefix, model_name)

Processing ..\Data\Relationships No Keywords\intra_goalGLOBAL_SGD2023_gemma3-4b.csv ...
  Processed 0/22914 rows...


In [ ]:
process_rel_path = "..\\Data\\Relationships No Keywords\\"
model_name="llama"

prefix = 'cross_goalGLOBAL_SGD2023_gemma3-4b'

classify_relationships(process_rel_path, prefix, model_name)